In [1]:
import numpy as np

In [2]:
import statsmodels.api as sm
import statsmodels.formula.api as smf

In [3]:
data = sm.datasets.get_rdataset("dietox", "geepack").data
data.head(26)

,Pig,Evit,Cu,Litter,Start,Weight,Feed,Time
0,4601,Evit000,Cu000,1,26.50000,26.50000,NaN,1
1,4601,Evit000,Cu000,1,26.50000,27.59999,5.200005,2
2,4601,Evit000,Cu000,1,26.50000,36.50000,17.600000,3
3,4601,Evit000,Cu000,1,26.50000,40.29999,28.500000,4
4,4601,Evit000,Cu000,1,26.50000,49.09998,45.200001,5
5,4601,Evit000,Cu000,1,26.50000,55.39999,56.900002,6
6,4601,Evit000,Cu000,1,26.50000,59.59998,71.700005,7
7,4601,Evit000,Cu000,1,26.50000,67.00000,86.800001,8
8,4601,Evit000,Cu000,1,26.50000,76.59998,104.900002,9
9,4601,Evit000,Cu000,1,26.50000,86.50000,123.000000,10


In [4]:
print(len(data), len(data.Pig.unique()))

861 72


---

## Growth curves of pigs
These are longitudinal data from a factorial experiment. The outcome variable is the weight of each pig, and the only predictor variable we will use here is “time”. First we fit a model that expresses the mean weight as a linear function of time, with a random intercept for each pig. The model is specified using formulas. Since the random effects structure is not specified, the default random effects structure (a random intercept for each group) is automatically used.

In [5]:
md = smf.mixedlm("Weight ~ Time", data, groups=data["Pig"])
mdf = md.fit(method=["lbfgs"])
print(mdf.summary())

         Mixed Linear Model Regression Results
Model:            MixedLM Dependent Variable: Weight    
No. Observations: 861     Method:             REML      
No. Groups:       72      Scale:              11.3669   
Min. group size:  11      Log-Likelihood:     -2404.7753
Max. group size:  12      Converged:          Yes       
Mean group size:  12.0                                  
--------------------------------------------------------
             Coef.  Std.Err.    z    P>|z| [0.025 0.975]
--------------------------------------------------------
Intercept    15.724    0.788  19.952 0.000 14.179 17.268
Time          6.943    0.033 207.939 0.000  6.877  7.008
Group Var    40.395    2.149                            



Note that in the statsmodels summary of results, the fixed effects and random effects parameter estimates are shown in a single table. The random effect for animal is labeled “Intercept RE” in the statsmodels output above.

There has been a lot of debate about whether the standard errors for random effect variance and covariance parameters are useful. While there is good reason to question their utility, we elected to include the standard errors in the summary table, but do not show the corresponding Wald confidence intervals.

---

Next we fit a model with two random effects for each animal: a random intercept, and a random slope (with respect to time). This means that each pig may have a different baseline weight, as well as growing at a different rate. The formula specifies that “Time” is a covariate with a random coefficient. By default, formulas always include an intercept (which could be suppressed here using “0 + Time” as the formula).

In [6]:
md = smf.mixedlm("Weight ~ Time", data, groups=data["Pig"], re_formula="~Time")
mdf = md.fit(method=["lbfgs"])
print(mdf.summary())

           Mixed Linear Model Regression Results
Model:             MixedLM  Dependent Variable:  Weight    
No. Observations:  861      Method:              REML      
No. Groups:        72       Scale:               6.0372    
Min. group size:   11       Log-Likelihood:      -2217.0475
Max. group size:   12       Converged:           Yes       
Mean group size:   12.0                                    
-----------------------------------------------------------
                 Coef.  Std.Err.   z    P>|z| [0.025 0.975]
-----------------------------------------------------------
Intercept        15.739    0.550 28.603 0.000 14.660 16.817
Time              6.939    0.080 86.925 0.000  6.783  7.095
Group Var        19.503    1.561                           
Group x Time Cov  0.294    0.153                           
Time Var          0.416    0.033                           



The random intercept and random slope are only weakly correlated $0.294/\sqrt{19.493*0.416}\approx0.1$. So next we fit a model in which the two random effects are constrained to be uncorrelated:

In [7]:
0.294 / (19.493 * 0.416) ** 0.5

0.10324316832591753

In [8]:
md = smf.mixedlm("Weight ~ Time", data, groups=data["Pig"], re_formula="~Time")
free = sm.regression.mixed_linear_model.MixedLMParams.from_components(np.ones(2), np.eye(2))

mdf = md.fit(free=free, method=["lbfgs"])
print(mdf.summary())

           Mixed Linear Model Regression Results
Model:             MixedLM  Dependent Variable:  Weight    
No. Observations:  861      Method:              REML      
No. Groups:        72       Scale:               6.0283    
Min. group size:   11       Log-Likelihood:      -2217.3481
Max. group size:   12       Converged:           Yes       
Mean group size:   12.0                                    
-----------------------------------------------------------
                 Coef.  Std.Err.   z    P>|z| [0.025 0.975]
-----------------------------------------------------------
Intercept        15.739    0.554 28.388 0.000 14.652 16.825
Time              6.939    0.080 86.248 0.000  6.781  7.097
Group Var        19.837    1.571                           
Group x Time Cov  0.000    0.000                           
Time Var          0.423    0.033                           



The likelihood drops by $0.3$ when we fix the correlation parameter to $0$. Comparing $2 \times 0.3 = 0.6$ to the $\chi^2$ reference distribution with 1 df suggests that the data are very consistent with a model in which this parameter is equal to $0$.